# Knowledge Distillation
- The concept of **knowledge distillation** is to utilize class probabilities of a higher-capacity model (teacher) as soft targets of a smaller model (student)
- The implement processes can be divided into several stages:
  1. Finish the `ResNet()` classes
  2. Train the teacher model (ResNet50) and the student model (ResNet18) from scratch, i.e. **without KD**
  3. Define the `Distiller()` class and `loss_re()`, `loss_fe()` functions
  4. Train the student model **with KD** from the teacher model in two different ways, response-based and feature based distillation
  5. Comparison of student models w/ & w/o KD

## Setup

In [1]:
! pip install torchinfo

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, random_split
from torchinfo import summary
from tqdm import tqdm
import sys
import numpy as np
import math
import matplotlib.pyplot as plt
import os
from PIL import Image

In [3]:
torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

## Download dataset

In [4]:
validation_split = 0.2
batch_size = 128

# data augmentation and normalization
transform_train = transforms.Compose([
                    transforms.RandomCrop(32, padding=4),
                    transforms.RandomHorizontalFlip(),
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

transform_test = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# download dataset
train_and_val_dataset = torchvision.datasets.CIFAR10(
    root='dataset/',
    train=True,
    transform=transform_train,
    download=True
)

test_dataset = torchvision.datasets.CIFAR10(
    root='dataset/',
    train=False,
    transform=transform_test,
    download=True
)

# split train and validation dataset
train_size = int((1 - validation_split) * len(train_and_val_dataset))
val_size = len(train_and_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_and_val_dataset, [train_size, val_size])

# create dataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

test_num = len(test_dataset)
test_steps = len(test_loader)

100%|██████████| 170M/170M [00:06<00:00, 28.4MB/s]


## Create teacher and student models
### Define BottleNeck for ResNet50

In [5]:
class BottleNeck(nn.Module):
    expansion = 4

    def __init__(self, in_channel, out_channel, stride=1, downsample=None, **kwargs):
        super(BottleNeck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=out_channel, kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(in_channels=out_channel, out_channels=out_channel, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)
        self.conv3 = nn.Conv2d(in_channels=out_channel, out_channels=out_channel * self.expansion, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channel * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out += identity
        out = self.relu(out)

        return out

### Define Resifual Block

In [6]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channel, out_channel, stride=1, downsample=None, **kwargs):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=out_channel, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=out_channel, out_channels=out_channel, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

### Define ResNet Model

In [7]:
class ResNet(nn.Module):

    def __init__(self, block, blocks_num, num_classes=1000):
        super(ResNet, self).__init__()
        self.in_channel = 64

        self.conv1 = nn.Conv2d(3, self.in_channel, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_channel)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, blocks_num[0])
        self.layer2 = self._make_layer(block, 128, blocks_num[1], stride=2)
        self.layer3 = self._make_layer(block, 256, blocks_num[2], stride=2)
        self.layer4 = self._make_layer(block, 512, blocks_num[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def _make_layer(self, block, channel, block_num, stride=1):
        downsample = None
        if stride != 1 or self.in_channel != channel * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channel, channel * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(channel * block.expansion))

        layers = []
        layers.append(block(self.in_channel, channel, downsample=downsample, stride=stride))
        self.in_channel = channel * block.expansion

        for _ in range(1, block_num):
            layers.append(block(self.in_channel, channel))

        return nn.Sequential(*layers)

    def forward(self, x):
        # 1. Finish the forward pass and return the output layer as well as hidden features.
        # 2. The output layer and hidden features will be used later for distilling.
        # 3. You can refer to the ResNet structure illustration to finish it.
        feature1 = None
        feature2 = None
        feature3 = None
        feature4 = None

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        feature1 = x
        x = self.layer2(x)
        feature2 = x
        x = self.layer3(x)
        feature3 = x
        x = self.layer4(x)
        feature4 = x

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x, [feature1, feature2, feature3, feature4]

### Define ResNet50 and Resnet18

In [8]:
def resnet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

def resnet50(num_classes=10):
    return ResNet(BottleNeck, [3, 4, 6, 3], num_classes=num_classes)

## Teacher Model (ResNet50)

In [9]:
Teacher = resnet50(num_classes=10)  # commment out this line if loading trained teacher model
# Teacher = torch.load('Teacher.pt', weights_only=False)  # loading trained teacher model
Teacher = Teacher.to(device)

In [10]:
summary(Teacher)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            1,728
├─BatchNorm2d: 1-2                       128
├─ReLU: 1-3                              --
├─MaxPool2d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BottleNeck: 2-1                   --
│    │    └─Conv2d: 3-1                  4,096
│    │    └─BatchNorm2d: 3-2             128
│    │    └─Conv2d: 3-3                  36,864
│    │    └─BatchNorm2d: 3-4             128
│    │    └─Conv2d: 3-5                  16,384
│    │    └─BatchNorm2d: 3-6             512
│    │    └─ReLU: 3-7                    --
│    │    └─Sequential: 3-8              16,896
│    └─BottleNeck: 2-2                   --
│    │    └─Conv2d: 3-9                  16,384
│    │    └─BatchNorm2d: 3-10            128
│    │    └─Conv2d: 3-11                 36,864
│    │    └─BatchNorm2d: 3-12            128
│    │    └─Conv2d: 3-13               

## Student Model (ResNet18)

In [11]:
Student = resnet18(num_classes=10)  # commment out this line if loading trained student model
# Student = torch.load('Student.pt', weights_only=False)  # loading trained student model
Student = Student.to(device)

In [12]:
summary(Student)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            1,728
├─BatchNorm2d: 1-2                       128
├─ReLU: 1-3                              --
├─MaxPool2d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BasicBlock: 2-1                   --
│    │    └─Conv2d: 3-1                  36,864
│    │    └─BatchNorm2d: 3-2             128
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  36,864
│    │    └─BatchNorm2d: 3-5             128
│    └─BasicBlock: 2-2                   --
│    │    └─Conv2d: 3-6                  36,864
│    │    └─BatchNorm2d: 3-7             128
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  36,864
│    │    └─BatchNorm2d: 3-10            128
├─Sequential: 1-6                        --
│    └─BasicBlock: 2-3                   --
│    │    └─Conv2d: 3-11                 73,728

## Define training function

In [13]:
def train_from_scratch(model, train_loader, val_loader, epochs, learning_rate, device, model_name):
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=learning_rate)

    loss = []
    train_error=[]
    val_error = []
    valdation_error = []
    train_loss = []
    valdation_loss = []
    train_accuraacy = []
    valdation_accuracy= []

    for epoch in range(epochs):
        train_loss = 0.0
        valid_loss = 0.0
        train_acc = 0.0
        valid_acc = 0.0
        correct = 0.
        total = 0.
        V_correct = 0.
        V_total = 0.

        model.train()
        train_bar = tqdm(train_loader, file=sys.stdout)
        for step, data in enumerate(train_bar):
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, hidden = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            pred = logits.data.max(1, keepdim=True)[1]
            correct += np.sum(np.squeeze(pred.eq(labels.data.view_as(pred))).cpu().numpy())
            total += images.size(0)
            train_acc =  correct/total
            train_bar.desc = "train epoch[{}/{}]".format(epoch + 1, epochs)

        model.eval()
        with torch.no_grad():
            val_bar = tqdm(val_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_labels = val_data
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                outputs, hidden_outputs = model(val_images)
                loss = criterion(outputs, val_labels)
                valid_loss += loss.item() * val_images.size(0)
                pred = outputs.data.max(1, keepdim=True)[1]
                V_correct += np.sum(np.squeeze(pred.eq(val_labels.data.view_as(pred))).cpu().numpy())
                V_total += val_images.size(0)
                val_bar.desc = "valid epoch[{}/{}]".format(epoch + 1, epochs)

        train_loss = train_loss / len(train_loader.dataset)
        train_error.append(train_loss)
        valid_loss = valid_loss / len(val_loader.dataset)
        val_error.append(valid_loss)
        train_accuraacy.append( correct / total)
        valdation_accuracy.append(V_correct / V_total)

        print('\tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(train_loss, valid_loss))
        print('\tTrain Accuracy: %.3fd%% (%2d/%2d)\tValdation Accuracy: %.3fd%% (%2d/%2d) '% (100. * correct / total, correct, total, 100. * V_correct / V_total, V_correct, V_total))

    torch.save(model, f'{model_name}.pt')
    print(f'{model_name}.pt is saved')

    print('Finished Training')

## Define testing function

In [14]:
def test(model, test_loader ,device, type=None):
    criterion = nn.CrossEntropyLoss()
    acc = 0.0
    test_loss = 0.0

    if type == None:
        model.eval()
    elif type == 'distiller':
        model.eval()
        model.teacher.eval()
        model.student.eval()
    else:
       raise ValueError(f'Error: only support response-based and feature-based distillation')

    with torch.no_grad():
        test_bar = tqdm(test_loader, file=sys.stdout)
        for test_data in test_bar:
            test_images, test_labels = test_data
            test_images, test_labels = test_images.to(device), test_labels.to(device)
            if type == None:
                outputs, features = model(test_images)
                loss = criterion(outputs, test_labels)
            elif type == 'distiller':
                outputs, loss = model(test_images, test_labels)
            else:
                raise ValueError(f'Error: only support response-based and feature-based distillation')

            predict_y = torch.max(outputs, dim=1)[1]
            acc += torch.eq(predict_y, test_labels.to(device)).sum().item()
            test_loss += loss.item()
            test_bar.desc = "test"

    test_accurate = acc / test_num
    print('test_loss: %.3f  test_accuracy: %.3f' %(test_loss / test_steps, test_accurate * 100))
    return test_loss / test_steps, test_accurate * 100.

## Train Teacher and Student model from scratch

In [15]:
# Decide the epochs and learning rate
train_from_scratch(Teacher, train_loader, val_loader, epochs=30, learning_rate=1e-3, device=device, model_name="Teacher")

valid epoch[1/30]: 100%|██████████| 79/79 [00:07<00:00, 11.28it/s]
	Training Loss: 1.681867 	Validation Loss: 1.461343
	Train Accuracy: 39.468d% (15787/40000)	Valdation Accuracy: 47.350d% (4735/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:07<00:00, 10.91it/s]
	Training Loss: 1.239514 	Validation Loss: 1.172891
	Train Accuracy: 56.035d% (22414/40000)	Valdation Accuracy: 59.360d% (5936/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:07<00:00, 11.20it/s]
	Training Loss: 1.031177 	Validation Loss: 1.104146
	Train Accuracy: 63.807d% (25523/40000)	Valdation Accuracy: 63.180d% (6318/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:07<00:00, 11.08it/s]
	Training Loss: 0.880516 	Validation Loss: 0.909590
	Train Accuracy: 69.495d% (27798/40000)	Valdation Accuracy: 67.500d% (6750/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:07<00:00, 11.09it/s]
	Training Loss: 0.773441 	Validation Loss: 0.848570
	Train Accuracy: 72.968d% (29187/40000)	Valdation Accuracy: 71.740d% (717

In [16]:
T_loss, T_accuracy = test(Teacher, test_loader, device=device)

test: 100%|██████████| 79/79 [00:08<00:00,  9.84it/s]
test_loss: 0.410  test_accuracy: 86.820


In [17]:
# Decide the epochs and learning rate
train_from_scratch(Student, train_loader, val_loader, epochs=30, learning_rate=1e-3, device=device, model_name="Student")

valid epoch[1/30]: 100%|██████████| 79/79 [00:05<00:00, 15.09it/s]
	Training Loss: 1.403443 	Validation Loss: 1.393013
	Train Accuracy: 48.590d% (19436/40000)	Valdation Accuracy: 52.530d% (5253/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:05<00:00, 13.52it/s]
	Training Loss: 1.019172 	Validation Loss: 0.943825
	Train Accuracy: 63.822d% (25529/40000)	Valdation Accuracy: 67.000d% (6700/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:06<00:00, 13.16it/s]
	Training Loss: 0.838638 	Validation Loss: 0.834221
	Train Accuracy: 70.627d% (28251/40000)	Valdation Accuracy: 71.210d% (7121/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:05<00:00, 13.56it/s]
	Training Loss: 0.722043 	Validation Loss: 0.827099
	Train Accuracy: 74.793d% (29917/40000)	Valdation Accuracy: 72.260d% (7226/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:05<00:00, 15.20it/s]
	Training Loss: 0.634595 	Validation Loss: 0.705529
	Train Accuracy: 78.145d% (31258/40000)	Valdation Accuracy: 75.750d% (757

In [18]:
S_loss, S_accuracy = test(Student, test_loader, device=device)

test: 100%|██████████| 79/79 [00:04<00:00, 17.66it/s]
test_loss: 0.497  test_accuracy: 85.910


## Define distillation

### Define the loss functions

In [19]:
# Finish the loss function for response-based distillation.
def loss_re(student_logits, teacher_logits, labels):
    T = 5.0 # Set temperature parameter
    alpha = 0.5 # Set weighting parameter

    # Implement loss calculation
    # Hard Loss 學習真實標籤
    hard_loss = F.cross_entropy(student_logits, labels)

    # Soft Loss 模仿老師的輸出分佈
    soft_loss = nn.KLDivLoss(reduction='batchmean')(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1)
    ) * (T * T)

    loss = (1. - alpha) * hard_loss + alpha * soft_loss

    return loss

In [20]:
# Finish the loss function for feature-based distillation.
def loss_fe(student_features, teacher_features, student_logits, labels):
    # Feature Loss 模仿中間層特徵
    feature_loss = 0.0
    for s_feat, t_feat in zip(student_features, teacher_features):
        feature_loss += F.mse_loss(s_feat, t_feat)

    # Task Loss 學習正確分類
    # 這是為了訓練 Student 的最後一層 (FC Layer)
    task_loss = F.cross_entropy(student_logits, labels)

    # Total Loss: 結合兩者
    alpha = 0.01
    loss = task_loss + alpha * feature_loss

    return loss

### Define Distillation Framework

In [21]:
class Distiller(nn.Module):
    def __init__(self, teacher, student, type):
        super(Distiller, self).__init__()

        # 1. Finish the __init__ method.
        self.teacher = teacher
        self.student = student
        self.type = type

        # student 的通道要跟 teacher 一樣
        if self.type == 'feature':
            self.adaptation_layers = nn.ModuleList([
                nn.Conv2d(64, 256, kernel_size=1), # 64 -> 256
                nn.Conv2d(128, 512, kernel_size=1), # 128 -> 512
                nn.Conv2d(256, 1024, kernel_size=1), # 256 -> 1024
                nn.Conv2d(512, 2048, kernel_size=1) # 512 -> 2048
            ])

    def forward(self, x, target):
        # 2. Finish the forward pass.

        student_logits, student_features = self.student(x) # 這裡會接收 ResNet 回傳的 [f1, f2, f3, f4]
        with torch.no_grad():
            teacher_logits, teacher_features = self.teacher(x)

        if self.type == 'response':
            loss_distill = loss_re(student_logits, teacher_logits, target) # call the loss_re()
        elif self.type == 'feature':
            # 將 Student 的每個特徵層通過對應的 adaptation layer
            transformed_features = []
            for i, layer in enumerate(self.adaptation_layers):
                # ResNet 裡的 feature1, feature2...
                transformed_features.append(layer(student_features[i]))

            loss_distill = loss_fe(transformed_features, teacher_features, student_logits, target)
        else:
            raise ValueError(f'Error: only support response-based and feature-based distillation')

        return student_logits, loss_distill

### Training function

In [22]:
def train_distillation(distiller, student, train_loader, val_loader, epochs, learning_rate, device):
    ce_loss = nn.CrossEntropyLoss()
    # define the parameter the optimizer used
    optimizer = torch.optim.Adam(distiller.parameters(), lr=learning_rate)

    loss = []
    train_error=[]
    val_error = []
    valdation_error = []
    train_loss = []
    valdation_loss = []
    train_accuraacy = []
    valdation_accuracy= []

    for epoch in range(epochs):
        distiller.train()
        distiller.teacher.train()
        distiller.student.train()

        train_loss = 0.0
        valid_loss = 0.0
        train_acc = 0.0
        valid_acc  = 0.0
        correct = 0.
        total = 0.
        V_correct = 0.
        V_total = 0.
        train_bar = tqdm(train_loader, file=sys.stdout)
        for step, data in enumerate(train_bar):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            outputs, loss = distiller(images, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            pred = outputs.data.max(1, keepdim=True)[1]
            result = pred.eq(labels.data.view_as(pred))
            result = np.squeeze(result.cpu().numpy())
            correct += np.sum(result)
            total += images.size(0)
            train_bar.desc = "train epoch[{}/{}]".format(epoch + 1, epochs)

        distiller.eval()
        distiller.teacher.eval()
        distiller.student.eval()

        with torch.no_grad():
            val_bar = tqdm(val_loader, file=sys.stdout)
            for val_data in val_bar:

                val_images, val_labels = val_data
                val_images, val_labels = val_images.to(device), val_labels.to(device)

                outputs, loss = distiller(val_images, val_labels)

                valid_loss += loss.item() * val_images.size(0)
                pred = outputs.max(1, keepdim=True)[1]
                V_correct += np.sum(np.squeeze(pred.eq(val_labels.data.view_as(pred))).cpu().numpy())
                V_total += val_images.size(0)
                val_bar.desc = "valid epoch[{}/{}]".format(epoch + 1, epochs)

        train_loss = train_loss / len(train_loader.dataset)
        train_error.append(train_loss)
        valid_loss = valid_loss / len(val_loader.dataset)
        val_error.append(valid_loss)
        train_accuraacy.append( correct / total)
        valdation_accuracy.append(V_correct / V_total)

        print('\tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(train_loss, valid_loss))
        print('\tTrain Accuracy: %.3fd%% (%2d/%2d)\tValdation Accuracy: %.3fd%% (%2d/%2d) '% (100. * correct / total, correct, total, 100. * V_correct / V_total, V_correct, V_total))

    print('Finished Distilling')

## Response-based distillation

In [23]:
# Decide the epochs and learning rate
Student_re = resnet18(num_classes=10)
Student_re = Student_re.to(device)
distiller_re = Distiller(Teacher, Student_re, type='response')
train_distillation(distiller_re, Student_re, train_loader, val_loader, epochs=30, learning_rate=1e-3, device=device)

valid epoch[1/30]: 100%|██████████| 79/79 [00:07<00:00, 10.05it/s]
	Training Loss: 3.940805 	Validation Loss: 2.820513
	Train Accuracy: 51.203d% (20481/40000)	Valdation Accuracy: 62.320d% (6232/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:07<00:00, 10.59it/s]
	Training Loss: 2.082545 	Validation Loss: 2.225618
	Train Accuracy: 68.918d% (27567/40000)	Valdation Accuracy: 67.250d% (6725/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:07<00:00, 10.86it/s]
	Training Loss: 1.470155 	Validation Loss: 1.311291
	Train Accuracy: 75.597d% (30239/40000)	Valdation Accuracy: 76.210d% (7621/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:07<00:00, 10.36it/s]
	Training Loss: 1.163015 	Validation Loss: 1.216785
	Train Accuracy: 78.897d% (31559/40000)	Valdation Accuracy: 77.860d% (7786/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:07<00:00, 10.15it/s]
	Training Loss: 0.982940 	Validation Loss: 1.065892
	Train Accuracy: 81.362d% (32545/40000)	Valdation Accuracy: 79.420d% (794

In [24]:
reS_loss, reS_accuracy = test(distiller_re, test_loader, type='distiller', device=device)

test: 100%|██████████| 79/79 [00:05<00:00, 13.42it/s]
test_loss: 0.560  test_accuracy: 87.890


## Feature-based distillation

In [25]:
# Decide the epochs and learning rate
Student_fe = resnet18(num_classes=10)
Student_fe = Student_fe.to(device)
distiller_fe = Distiller(Teacher, Student_fe, type='feature')

# Fix: Move the adaptation layers to the device if feature distillation is used
# (Ideally, this should be done in the Distiller's __init__ method)
if distiller_fe.type == 'feature':
    distiller_fe.adaptation_layers.to(device)

train_distillation(distiller_fe, Student_fe, train_loader, val_loader, epochs=30 , learning_rate=1e-3 , device=device)

valid epoch[1/30]: 100%|██████████| 79/79 [00:07<00:00, 10.07it/s]
	Training Loss: 1.473392 	Validation Loss: 1.419115
	Train Accuracy: 48.325d% (19330/40000)	Valdation Accuracy: 49.980d% (4998/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:07<00:00, 10.24it/s]
	Training Loss: 1.043501 	Validation Loss: 1.001586
	Train Accuracy: 64.770d% (25908/40000)	Valdation Accuracy: 66.370d% (6637/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:07<00:00, 10.50it/s]
	Training Loss: 0.855617 	Validation Loss: 0.948159
	Train Accuracy: 71.453d% (28581/40000)	Valdation Accuracy: 69.330d% (6933/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:07<00:00, 10.42it/s]
	Training Loss: 0.749661 	Validation Loss: 0.755526
	Train Accuracy: 75.787d% (30315/40000)	Valdation Accuracy: 75.420d% (7542/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:07<00:00, 10.56it/s]
	Training Loss: 0.676407 	Validation Loss: 0.820339
	Train Accuracy: 78.168d% (31267/40000)	Valdation Accuracy: 73.330d% (733

In [26]:
ftS_loss, ftS_accuracy = test(distiller_fe, test_loader, type='distiller', device=device)

test: 100%|██████████| 79/79 [00:06<00:00, 12.69it/s]
test_loss: 0.473  test_accuracy: 87.790


## Result and Comparison

In [27]:
print(f'Teacher from scratch: loss = {T_loss:.2f}, accuracy = {T_accuracy:.2f}')
print(f'Student from scratch: loss = {S_loss:.2f}, accuracy = {S_accuracy:.2f}')
print(f'Response-based student: loss = {reS_loss:.2f}, accuracy = {reS_accuracy:.2f}')
print(f'Featured-based student: loss = {ftS_loss:.2f}, accuracy = {ftS_accuracy:.2f}')

Teacher from scratch: loss = 0.41, accuracy = 86.82
Student from scratch: loss = 0.50, accuracy = 85.91
Response-based student: loss = 0.56, accuracy = 87.89
Featured-based student: loss = 0.47, accuracy = 87.79
